### SectionB–Bronze ingestion

In [0]:
dbutils.widgets.text("catalog", "vp_ce_2026_07_1382", "Catalog Name")
catalog = dbutils.widgets.get("catalog")

In [0]:
# B1: Load stations.csv

spark.sql(f"""
CREATE TABLE IF NOT EXISTS IDENTIFIER('{catalog}.bronze.stations') (
    _source_file STRING,
    _ingested_at TIMESTAMP
)
""")

spark.sql(f"""
COPY INTO IDENTIFIER('{catalog}.bronze.stations')
FROM (
    SELECT *,
           _metadata.file_path AS _source_file,
           current_timestamp() AS _ingested_at
    FROM '/Volumes/vp_ce_2026_07_1382/landing/raw/stations/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
# B2: Load tariffs.csv

spark.sql(f"""
CREATE TABLE IF NOT EXISTS IDENTIFIER('{catalog}.bronze.tariffs') (
    _source_file STRING,
    _ingested_at TIMESTAMP
)
""")

spark.sql(f"""
COPY INTO IDENTIFIER('{catalog}.bronze.tariffs')
FROM (
    SELECT *, 
           _metadata.file_path AS _source_file, 
           current_timestamp() AS _ingested_at 
    FROM '/Volumes/vp_ce_2026_07_1382/landing/raw/tariffs/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
# B2: Load members.csv

spark.sql(f"""
CREATE TABLE IF NOT EXISTS IDENTIFIER('{catalog}.bronze.members') (
    _source_file STRING,
    _ingested_at TIMESTAMP,
    change_effective_date DATE
)
""")

spark.sql(f"""
COPY INTO IDENTIFIER('{catalog}.bronze.members')
FROM (
    SELECT *, 
           _metadata.file_path AS _source_file, 
           current_timestamp() AS _ingested_at
    FROM '/Volumes/vp_ce_2026_07_1382/landing/raw/members/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
# B3: Load chargers.json

spark.sql(f"""
CREATE TABLE IF NOT EXISTS IDENTIFIER('{catalog}.bronze.chargers') (
    _source_file STRING,
    _ingested_at TIMESTAMP
)
""")

spark.sql(f"""
COPY INTO IDENTIFIER('{catalog}.bronze.chargers')
FROM (
    SELECT *,
           _metadata.file_path AS _source_file, 
           current_timestamp() AS _ingested_at 
    FROM '/Volumes/vp_ce_2026_07_1382/landing/raw/chargers/'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true', 'multiline' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
from pyspark.sql.functions import *

# B4: Load sessions.json

df_orders = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("header", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") 
    .option("cloudFiles.inferColumnTypes", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .option("cloudFiles.schemaLocation", "/Volumes/vp_ce_2026_07_1382/ops/pipeline_state/sessions_schema/")
    .load("/Volumes/vp_ce_2026_07_1382/landing/raw/sessions/")
    .select(
        col("*"),
        col("_metadata.file_path").alias("_source_file"),
        current_timestamp().alias("_ingested_at")
    )
)

(
    df_orders.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/vp_ce_2026_07_1382/ops/pipeline_state/sessions_checkpoints/")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .table(f"{catalog}.bronze.sessions")
)